# Europe: job vacancy rates by activity

Eurostat quarterly job vacancy rate (vacancies as a share of occupied plus vacant posts) by activity, splicing the closed NACE Rev. 2 table (to 2025) with the NACE Rev. 2.1 table; where both report a quarter, Rev. 2.1 is kept.

**Datasets:** `eurostat/jvs_q_r21`, `eurostat/jvs_q_nace2`

This notebook reads the latest vintage each time it runs; re-run after a new release to update every table and chart.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
from labobs.load import load, vintage, curated, raw_payload
from labobs.nb import select, pivot, latest, plot_lines, plot_bars, to_annual, FOCUS, FOCUS_NAMES
pd.set_option("display.max_rows", 80)
pd.set_option("display.float_format", "{:,.2f}".format)

In [2]:
DATASETS = ["eurostat/jvs_q_r21", "eurostat/jvs_q_nace2"]
for k in DATASETS:
    v = vintage(k)
    print(f"{k:50} vintage {v.get('vintage', 'not yet ingested'):12} sha {v.get('sha256', '')[:12]}")

eurostat/jvs_q_r21                                 vintage not yet ingested sha 
eurostat/jvs_q_nace2                               vintage not yet ingested sha 


In [3]:
v = pd.concat([load("eurostat/jvs_q_nace2").assign(nace_rev="2"), load("eurostat/jvs_q_r21").assign(nace_rev="2.1")])
v = v.sort_values("nace_rev").drop_duplicates([c for c in v.columns if c not in ("value", "nace_rev", "dataset", "obs_flag")], keep="last")
print({c: v[c].drop_duplicates().head(15).tolist() for c in ["indic_em", "nace_r2", "sizeclas"] if c in v.columns})

FileNotFoundError: eurostat/jvs_q_nace2: no processed data yet (/home/runner/work/Labour-Market-Policy/Labour-Market-Policy/data/vacancies/eurostat/jvs_q_nace2/latest.csv). Run `labobs ingest --source eurostat --dataset jvs_q_nace2` or wait for the daily run.

In [4]:
sel = dict(indic_em="JOBRATE")
if "sizeclas" in v.columns: sel["sizeclas"] = "TOTAL"
r = select(v, **sel)
act = {"B-S": "Industry, construction & services", "F": "Construction", "Q": "Health & social work", "J": "Information & communication", "C": "Manufacturing", "P": "Education", "D": "Energy"}
ea = select(r, ref_area="EU27_2020", nace_r2=list(act))
plot_lines(pivot(ea, columns="nace_r2").rename(columns=act).loc["2015":], "EU27 job vacancy rate by activity", "%", "Eurostat jvs_q_nace2", names={});

AttributeError: 'dict' object has no attribute 'columns'

## Curator notes

_Written by the curator after each new vintage: what changed, which revisions matter, and what to watch. See `DATA_CHANGELOG.md` for the full log._